In [42]:
### Importing required library 
import numpy as np 
from rdkit.Chem import AllChem
import pickle
import pandas as pd
from rdkit import Chem
import xgboost
import base64
import pickle
import numpy as np
import pandas as pd
from rdkit import Chem,DataStructs
from rdkit.Chem import MolFromSmiles, Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors
from rdkit.Chem import Descriptors
from rdkit.Chem import Lipinski
from rdkit.Chem import Crippen
import base64
import shap
from xgboost import XGBRegressor
import xgboost as xgb
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import rdMolDescriptors
import requests
from bs4 import BeautifulSoup
import deepchem as dc
from sklearn.preprocessing import StandardScaler

In [81]:
import pandas as pd 
df_set1 = pd.read_excel('JCIM_test.xlsx', sheet_name="SET1")
df_set2 = pd.read_excel('JCIM_test.xlsx', sheet_name="SET2")

In [82]:
print(df_set1.shape)
print(df_set2.shape)

(100, 3)
(32, 3)


In [16]:
train_set =pd.read_csv('unique_train4_new26.csv')

In [67]:
from rdkit.Chem import MolFromSmiles as smi2mol
from rdkit.Chem import MolToSmiles as mol2smi

## Function to create canonical smiles 
def canon(smi):
    try:
        mol=smi2mol(smi, sanitize=True)
        smi_canon=mol2smi(mol, isomericSmiles=False, canonical=True)
        return(smi_canon)
    except:
        print("ERROR")
        return(smi)

In [83]:
df_set1['smiles_canon'] = [canon(smi) for smi in df_set1.SMILES]
df_set2['smiles_canon'] = [canon(smi) for smi in df_set2.SMILES]

In [20]:
### Finding the match between train and test data 
Match_rows_set1 = pd.merge(df_set1, train_set, on=['smiles_canon'], how='inner')
#mergedStuff.head()
print(len(Match_rows_set1))

37


In [28]:
### Finding the matching smiles in the train dataset..and removing it 
cond = train_set['smiles_canon'].isin(Match_rows_set1['smiles_canon'])

#### Dropping the smiles which is same and making dataframe with uniques smiles and no smiles same in ttrain  dataset 
train_set.drop(train_set[cond].index, inplace = True)

In [29]:
train_set.shape

(17900, 7)

In [30]:
### Match with remaining train set 
Match_rows_set2 = pd.merge(df_set2, train_set, on=['smiles_canon'], how='inner')
#mergedStuff.head()
print(len(Match_rows_set2))

0


In [25]:
### Now there is no data in train set which match with df_set1 or df_set2 

In [34]:
### After removing the data match with train data 
print(df_set1.shape)
print(df_set2.shape) 

(100, 3)
(32, 3)


In [36]:
df_set1

,Sr No,COMPOUND_Name,SMILES
0,1,Acetazolamide,CC(NC1=NN=C(S1)[S](N)(=O)=O)=O
1,2,Acetylsalicylic_Acid,C(C)(=O)OC1=CC=CC=C1C(=O)O
2,3,Alclofenac,C=CCOc1ccc(cc1Cl)CC(=O)O
3,4,Ambroxol,O[C@@H]2CC[C@@H](NCc1cc(Br)cc(Br)c1N)CC2
4,5,Aripiprazole,O=C1Nc2c(ccc(OCCCCN3CCN(c4c(Cl)c(Cl)ccc4)CC3)c...
...,...,...,...
95,96,Thiacetazone,C1=C(C=CC(=C1)NC(=O)C)/C=N/NC(=S)N
96,97,Triamcinolone,[C@]23([C@H]([C@H]1[C@]([C@](C(CO)=O)(O)[C@@H]...
97,98,Triamterene,C3=C(C1=C(N=C2C(=N1)C(=NC(=N2)N)N)N)C=CC=C3
98,99,Warfarin,C3=C(C(C1=C(OC2=C(C1=O)C=CC=C2)O)CC(=O)C)C=CC=C3


In [40]:
#### All the function to create the descriptors 
import pandas as pd
def calculate_aromatic_proportion(smiles):
    # Parse SMILES string and generate molecular representation
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None  # Invalid SMILES
    
    # Identify aromatic atoms
    aromatic_atoms = [atom.GetAtomicNum() for atom in mol.GetAtoms() if atom.GetIsAromatic()]
    
    # Calculate aromatic proportion
    total_atoms = mol.GetNumAtoms()
    aromatic_proportion = len(aromatic_atoms) / total_atoms
    
    return aromatic_proportion
####  function for handling single smiles 
def calculate_rdkit_features(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if not mol:
        raise ValueError("Invalid SMILES string.")
    
    # List of RDKit descriptors
    descriptor_names = [desc[0] for desc in Descriptors._descList]
    descriptor_funcs = [desc[1] for desc in Descriptors._descList]
    
    # Compute descriptors
    features = [func(mol) for func in descriptor_funcs]
    
    # Create DataFrame
    df = pd.DataFrame([features], columns=descriptor_names)
    return df
#### Function for list of the smiles 
def calculate_rdkit_features1(smiles_list):
    """
    Calculate RDKit descriptors for a list of SMILES strings.

    Parameters:
        smiles_list (list of str): List of SMILES strings.

    Returns:
        pd.DataFrame: A DataFrame where each row corresponds to a SMILES string
                      and each column corresponds to a descriptor.
    """
    # List of RDKit descriptors
    descriptor_names = [desc[0] for desc in Descriptors._descList]
    descriptor_funcs = [desc[1] for desc in Descriptors._descList]

    data = []
    invalid_smiles = []

    for i, smiles in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(smiles)
        if not mol:
            invalid_smiles.append((i, smiles))
            continue
        
        # Compute descriptors
        features = [func(mol) for func in descriptor_funcs]
        data.append(features)

    # Create DataFrame
    df = pd.DataFrame(data, columns=descriptor_names, index=[i for i, smiles in enumerate(smiles_list) if smiles not in dict(invalid_smiles)])

    # Log invalid SMILES strings
    if invalid_smiles:
        print(f"Invalid SMILES found at indices: {[idx for idx, sm in invalid_smiles]}\nInvalid SMILES: {[sm for idx, sm in invalid_smiles]}")

    return df
### function for single smiles and list of the smiles 
def fingerprint1(smiles, r, n): 
    # Check if input is a single SMILES string or a list
    if isinstance(smiles, str):
        smiles = [smiles]  # Convert single SMILES string to a list

    # Convert SMILES strings to RDKit molecules
    mols = [Chem.MolFromSmiles(SMILES_string) for SMILES_string in smiles]

    # Compute fingerprints
    fingerprints_array = []
    for m in mols:
        bi = {}
        fingerprint = rdMolDescriptors.GetMorganFingerprintAsBitVect(m, radius=r, nBits=n, bitInfo=bi)
        array = np.zeros((1,), dtype=int)
        DataStructs.ConvertToNumpyArray(fingerprint, array)
        fingerprints_array.append(array)

    # Convert to a DataFrame with bits as columns
    fingerprints_df = pd.DataFrame(fingerprints_array)

    # Return the DataFrame for a single SMILES
    return fingerprints_df
def get_charges(smiles):
    if '+' in smiles:
        return 1
    elif '-' in smiles:
        return -1
    else:
        return 0

def get_many_double_bonds(smiles):
    mol = Chem.MolFromSmiles(smiles)
    double_bond_count = sum(1 for bond in mol.GetBonds() if bond.GetBondType() == Chem.rdchem.BondType.DOUBLE)
    return 1 if double_bond_count > 4 else 0

def get_atom_degrees(smiles):
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    bonds = mol.GetBonds()
    sum_degree_vector = np.zeros(7)
    for bond in bonds:
        atom1 = bond.GetBeginAtom()
        atom2 = bond.GetEndAtom()
        atom_degree_vector = np.array([1 if atom1.GetDegree() == d else 0 for d in range(7)])
        sum_degree_vector += atom_degree_vector
        atom_degree_vector = np.array([1 if atom2.GetDegree() == d else 0 for d in range(7)])
        sum_degree_vector += atom_degree_vector
    return sum_degree_vector.astype(int)

def get_atom_valences(smiles):
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    bonds = mol.GetBonds()
    sum_valence_vector = np.zeros(7)
    for bond in bonds:
        atom1 = bond.GetBeginAtom()
        atom2 = bond.GetEndAtom()
        atom_valence_vector = np.array([1 if atom1.GetTotalValence() == v else 0 for v in range(7)])
        sum_valence_vector += atom_valence_vector
        atom_valence_vector = np.array([1 if atom2.GetTotalValence() == v else 0 for v in range(7)])
        sum_valence_vector += atom_valence_vector
    return sum_valence_vector.astype(int)

def get_atom_hybridization(smiles):
    hybridizations = [
        Chem.rdchem.HybridizationType.S,
        Chem.rdchem.HybridizationType.SP, 
        Chem.rdchem.HybridizationType.SP2, 
        Chem.rdchem.HybridizationType.SP3, 
        Chem.rdchem.HybridizationType.SP3D, 
        Chem.rdchem.HybridizationType.SP3D2, 
        Chem.rdchem.HybridizationType.UNSPECIFIED]
    
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    bonds = mol.GetBonds()
    sum_hybrid_vector = np.zeros(7)
    for bond in bonds:
        atom1 = bond.GetBeginAtom()
        atom2 = bond.GetEndAtom()
        atom_hybrid_vector = np.array([1 if atom1.GetHybridization() == h else 0 for h in hybridizations])
        sum_hybrid_vector += atom_hybrid_vector
        atom_hybrid_vector = np.array([1 if atom2.GetHybridization() == h else 0 for h in hybridizations])
        sum_hybrid_vector += atom_hybrid_vector
    return sum_hybrid_vector.astype(int)

def get_aromatic_atoms(smiles):
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    bonds = mol.GetBonds()
    sum_aromatic_vector = np.zeros(1)
    for bond in bonds:
        atom1 = bond.GetBeginAtom()
        atom2 = bond.GetEndAtom()
        atom_aromatic_vector = np.array([1 if atom1.GetIsAromatic() else 0])
        sum_aromatic_vector += atom_aromatic_vector
        atom_aromatic_vector = np.array([1 if atom2.GetIsAromatic() else 0])
        sum_aromatic_vector += atom_aromatic_vector
    return sum_aromatic_vector.astype(int)[0]

def get_bond_types(smiles):
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    bonds = mol.GetBonds()
    sum_bond_type_vector = np.zeros(5)
    for bond in bonds:
        bond_type = bond.GetBondType().name
        bond_type_vector = np.array([1 if t == bond_type else 0 for t in ['SINGLE', 'DOUBLE', 'TRIPLE', 'AROMATIC', 'ZERO']])
        sum_bond_type_vector += bond_type_vector
    return sum_bond_type_vector.astype(int)

def is_conjugated(smiles):
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    bonds = mol.GetBonds()
    sum_conjugated = np.zeros(1)
    for bond in bonds:
        is_conjugated = 1 if bond.GetIsConjugated() else 0
        conjugation_vector = np.array([is_conjugated])
        sum_conjugated += conjugation_vector
    return sum_conjugated.astype(int)[0]

def get_bonds_in_ring(smiles):
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    return len(Chem.GetSymmSSSR(mol))

def get_bond_chirality(smiles):
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    bonds = mol.GetBonds()
    sum_chirality = np.zeros(4)
    for bond in bonds:
        chirality = bond.GetStereo()
        chirality_vector = np.array([1 if chirality == c else 0 for c in [Chem.rdchem.BondStereo.STEREONONE,
                                                                          Chem.rdchem.BondStereo.STEREOANY,
                                                                          Chem.rdchem.BondStereo.STEREOZ,
                                                                          Chem.rdchem.BondStereo.STEREOE]])
        sum_chirality += chirality_vector
    return sum_chirality.astype(int)

def get_n_atoms(smiles):
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    return mol.GetNumAtoms()

def get_n_bonds(smiles):
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    return mol.GetNumBonds()

def get_n_rings(smiles):
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    return len(Chem.GetSymmSSSR(mol))
### Funntion for creating 38 features for single smiles 
def generate_features_single38(smiles):
    columns = [
        'charge', 'many_double_bonds', 'atoms_degree_0', 'atoms_degree_1',
        'atoms_degree_2', 'atoms_degree_3', 'atoms_degree_4', 'atoms_degree_5',
        'atoms_degree_6', 'atoms_valence_0', 'atoms_valence_1', 'atoms_valence_2',
        'atoms_valence_3', 'atoms_valence_4', 'atoms_valence_5', 'atoms_valence_6',
        'atom_hybridization_S', 'atom_hybridization_SP', 'atom_hybridization_SP2',
        'atom_hybridization_SP3', 'atom_hybridization_SP3D', 'atom_hybridization_SP3D2',
        'atom_hybridization_UNSPECIFIED', 'aromatic_atoms', 'single_bonds', 'double_bonds',
        'triple_bonds', 'aromatic_bonds', 'zero_bonds', 'conjugated_bonds', 'bonds_in_ring',
        'chirality_none', 'chirality_any', 'chirality_z', 'chirality_e', 'n_atoms',
        'n_bonds', 'n_rings'
    ]

    # Calculate features for the single SMILES string
    charge = get_charges(smiles)
    many_double_bonds = get_many_double_bonds(smiles)
    atom_degrees = get_atom_degrees(smiles).tolist()
    atom_valences = get_atom_valences(smiles).tolist()
    atom_hybridizations = get_atom_hybridization(smiles).tolist()
    aromatic_atoms = get_aromatic_atoms(smiles)
    bond_types = get_bond_types(smiles).tolist()
    conjugated_bonds = is_conjugated(smiles)
    bonds_in_ring = get_bonds_in_ring(smiles)
    bond_chirality = get_bond_chirality(smiles).tolist()
    n_atoms = get_n_atoms(smiles)
    n_bonds = get_n_bonds(smiles)
    n_rings = get_n_rings(smiles)

    # Combine all features into a single row
    feature_row = [
        charge, many_double_bonds, atom_degrees[0], atom_degrees[1],
        atom_degrees[2], atom_degrees[3], atom_degrees[4], atom_degrees[5],
        atom_degrees[6], atom_valences[0], atom_valences[1], atom_valences[2],
        atom_valences[3], atom_valences[4], atom_valences[5], atom_valences[6],
        atom_hybridizations[0], atom_hybridizations[1], atom_hybridizations[2],
        atom_hybridizations[3], atom_hybridizations[4], atom_hybridizations[5],
        atom_hybridizations[6], aromatic_atoms, bond_types[0], bond_types[1],
        bond_types[2], bond_types[3], bond_types[4], conjugated_bonds,
        bonds_in_ring, bond_chirality[0], bond_chirality[1], bond_chirality[2],
        bond_chirality[3], n_atoms, n_bonds, n_rings
    ]

    # Convert the feature row into a DataFrame
    return pd.DataFrame([feature_row], columns=columns)
#### function for creating the 38 features for list of the smiles 
def generate_features38(smiles_list):
    columns = [
        'charge', 'many_double_bonds', 'atoms_degree_0', 'atoms_degree_1',
        'atoms_degree_2', 'atoms_degree_3', 'atoms_degree_4', 'atoms_degree_5',
        'atoms_degree_6', 'atoms_valence_0', 'atoms_valence_1', 'atoms_valence_2',
        'atoms_valence_3', 'atoms_valence_4', 'atoms_valence_5', 'atoms_valence_6',
        'atom_hybridization_S', 'atom_hybridization_SP', 'atom_hybridization_SP2',
        'atom_hybridization_SP3', 'atom_hybridization_SP3D', 'atom_hybridization_SP3D2',
        'atom_hybridization_UNSPECIFIED', 'aromatic_atoms', 'single_bonds', 'double_bonds',
        'triple_bonds', 'aromatic_bonds', 'zero_bonds', 'conjugated_bonds', 'bonds_in_ring',
        'chirality_none', 'chirality_any', 'chirality_z', 'chirality_e', 'n_atoms',
        'n_bonds', 'n_rings'
    ]
    
    features = []
    for smiles in smiles_list:
        charge = get_charges(smiles)
        many_double_bonds = get_many_double_bonds(smiles)
        atom_degrees = get_atom_degrees(smiles).tolist()
        atom_valences = get_atom_valences(smiles).tolist()
        atom_hybridizations = get_atom_hybridization(smiles).tolist()
        aromatic_atoms = get_aromatic_atoms(smiles)
        bond_types = get_bond_types(smiles).tolist()
        conjugated_bonds = is_conjugated(smiles)
        bonds_in_ring = get_bonds_in_ring(smiles)
        bond_chirality = get_bond_chirality(smiles).tolist()
        n_atoms = get_n_atoms(smiles)
        n_bonds = get_n_bonds(smiles)
        n_rings = get_n_rings(smiles)

        feature_row = [
            charge, many_double_bonds, atom_degrees[0], atom_degrees[1],
            atom_degrees[2], atom_degrees[3], atom_degrees[4], atom_degrees[5],
            atom_degrees[6], atom_valences[0], atom_valences[1], atom_valences[2],
            atom_valences[3], atom_valences[4], atom_valences[5], atom_valences[6],
            atom_hybridizations[0], atom_hybridizations[1], atom_hybridizations[2],
            atom_hybridizations[3], atom_hybridizations[4], atom_hybridizations[5],
            atom_hybridizations[6], aromatic_atoms, bond_types[0], bond_types[1],
            bond_types[2], bond_types[3], bond_types[4], conjugated_bonds,
            bonds_in_ring, bond_chirality[0], bond_chirality[1], bond_chirality[2],
            bond_chirality[3], n_atoms, n_bonds, n_rings
        ]
        
        features.append(feature_row)
    
    return pd.DataFrame(features, columns=columns)

### Function for 7 groups for single smiles and list of the smiles 
def get_functional_groups1(smiles):
    from rdkit import Chem
    import pandas as pd

    functional_groups = {
        # Polar functional groups
        'Hydroxyl Group': '[OH]',
        'Carbonyl Group': 'C=O',
        'Amide Group': 'C(=O)N',
        'Carboxyl Group': 'C(=O)[OH]',
        # Non-polar functional groups
        'Alkyl': '[R]', 
        'Aromatic Rings': 'c',
        'Alkene': 'C=C'
    }
    
    # Check if input is a single SMILES string
    if isinstance(smiles, str):
        smiles = [smiles]  # Wrap it in a list for consistency
    
    results = []
    for s in smiles:
        mol = Chem.MolFromSmiles(s)
        if mol:  # Ensure valid SMILES
            fg_presence = {fg: 1 if mol.HasSubstructMatch(Chem.MolFromSmarts(smarts)) else 0 for fg, smarts in functional_groups.items()}
        else:
            fg_presence = {fg: 0 for fg in functional_groups}  # Default to 0 if SMILES is invalid
        #fg_presence['SMILES'] = s
        results.append(fg_presence)
    
    # Convert results to a DataFrame
    data = pd.DataFrame(results)
    
    return data
### for single smiles 
def calc_mol_weight(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        return Descriptors.MolWt(mol)
    return None
#### for smiles list 
def calc_mol_weight1(smiles_list):
    """
    Calculate molecular weights for a list of SMILES strings.

    Parameters:
        smiles_list (list of str): List of SMILES strings.

    Returns:
        list: Molecular weights for valid SMILES strings, or None for invalid ones.
    """
    return [
        Descriptors.MolWt(Chem.MolFromSmiles(smiles)) if Chem.MolFromSmiles(smiles) else None
        for smiles in smiles_list
    ]


In [43]:
SMILES=train_set.smiles_canon

In [44]:
### Generate the descriptors of the train data 
df125_new=calculate_rdkit_features1(SMILES)
                
df125 = df125_new.iloc[:, :125]
                 
df38=generate_features38(SMILES)
                
df128=fingerprint1(SMILES,2,128)
                
df7=get_functional_groups1(SMILES)
                
combined_df = pd.concat([df125, df128, df7, df38], axis=1)

[14:49:15] WARNING: not removing hydrogen atom without neighbors
[14:49:15] WARNING: not removing hydrogen atom without neighbors
[14:49:15] WARNING: not removing hydrogen atom without neighbors
[14:49:15] WARNING: not removing hydrogen atom without neighbors
[14:49:15] WARNING: not removing hydrogen atom without neighbors
[14:49:15] WARNING: not removing hydrogen atom without neighbors
[14:49:15] WARNING: not removing hydrogen atom without neighbors
[14:49:15] WARNING: not removing hydrogen atom without neighbors
[14:49:15] WARNING: not removing hydrogen atom without neighbors
[14:49:15] WARNING: not removing hydrogen atom without neighbors
[14:49:15] WARNING: not removing hydrogen atom without neighbors
[14:49:15] WARNING: not removing hydrogen atom without neighbors
[14:49:16] WARNING: not removing hydrogen atom without neighbors
[14:49:16] WARNING: not removing hydrogen atom without neighbors
[14:49:16] WARNING: not removing hydrogen atom without neighbors
[14:49:16] WARNING: not r

In [49]:
y_train=train_set['LogS']

In [53]:
SMILES_test1=df_set1.smiles_canon
SMILES_test2=df_set2.smiles_canon

In [54]:
### Generate the descriptors of the test data for set1  
df125_new=calculate_rdkit_features1(SMILES_test1)
                
df125 = df125_new.iloc[:, :125]
                 
df38=generate_features38(SMILES_test1)
                
df128=fingerprint1(SMILES_test1,2,128)
                
df7=get_functional_groups1(SMILES_test1)
                
df_test1 = pd.concat([df125, df128, df7, df38], axis=1)

In [55]:
### Generate the descriptors of the test data for set1  
df125_new=calculate_rdkit_features1(SMILES_test2)
                
df125 = df125_new.iloc[:, :125]
                 
df38=generate_features38(SMILES_test2)
                
df128=fingerprint1(SMILES_test2,2,128)
                
df7=get_functional_groups1(SMILES_test2)
                
df_test2 = pd.concat([df125, df128, df7, df38], axis=1)

In [50]:
import xgboost as xgb
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import make_scorer, mean_squared_error
import numpy as np


In [51]:
### Training the model on train data 
xgboost_model = xgboost.XGBRegressor(learning_rate =0.01,n_estimators=2000,max_depth=8,min_child_weight=4,gamma=0,subsample=0.7,
                                           colsample_bytree=0.8,nthread=2,scale_pos_weight=1,seed=27)
xgboost_model.fit(combined_df,y_train)    


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=0, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.01, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=8, max_leaves=None,
             min_child_weight=4, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=2000, n_jobs=None, nthread=2,
             num_parallel_tree=None, ...)

In [56]:
### Predicting on test data set1 
pred_xgb1 = xgboost_model.predict(df_test1)
pred_xgb2 = xgboost_model.predict(df_test2)
### Calculating the evaluation matrix MAE , RMSE and R2
#xgb_4=utilities.get_errors1(y_test,pred_xgb,"XGB_4")

In [57]:
pred_xgb1

array([-1.2859375 , -1.6400429 , -2.9879606 , -3.66207   , -6.012181  ,
       -6.0276036 , -3.419471  , -2.0507312 , -2.5989404 , -3.9784272 ,
       -3.7203546 , -7.4884825 , -4.0628543 , -3.6632705 , -2.6975358 ,
       -2.4553952 , -2.2646518 , -4.8680787 , -2.4785857 , -2.8368788 ,
       -5.260372  , -5.0862145 , -1.7801055 , -4.570567  , -4.5266433 ,
       -3.40636   , -4.0723557 , -3.951703  , -3.3367898 , -4.4484024 ,
       -4.530471  , -2.3366911 , -4.5934114 , -5.227117  , -3.5144255 ,
       -4.66348   , -3.4808052 ,  0.01662173, -3.5999734 , -4.1197453 ,
       -4.0513806 , -2.7922401 , -3.6807365 , -2.9905126 , -3.7897232 ,
       -3.201401  , -3.9543202 , -1.9342104 , -1.4839474 , -4.4443088 ,
       -4.4954467 , -4.739469  , -5.596032  , -4.048071  , -5.167198  ,
       -4.001347  , -5.2504625 , -5.144371  , -4.352373  , -4.152267  ,
       -3.7759821 , -6.9189997 , -4.2015157 , -3.610031  , -4.2384906 ,
       -2.9477808 , -4.5465913 , -3.425334  , -3.4744368 , -3.38

In [78]:
pred_xgb2

array([-1.3727082, -9.061896 , -5.5769315, -2.0957947, -6.067071 ,
       -5.7322245, -6.0249333, -6.2962823, -4.6979966, -5.5417185,
       -1.1303596, -4.168745 , -2.9769616, -2.5645869, -5.0063677,
       -4.9244456, -5.320439 , -9.048034 , -6.9912424, -5.9295673,
       -4.075219 , -4.8165474, -1.41531  , -3.196905 , -5.6930914,
       -7.7953095, -4.3014727, -2.67008  , -6.551431 , -6.9993668,
       -6.955107 , -3.5381866], dtype=float32)

In [84]:
df_set1['Prediction']=pred_xgb1
df_set2['Prediction']=pred_xgb2

In [86]:
df_set2

,Sr No,COMPOUND,SMILES,smiles_canon,Prediction
0,1,Amantadine,C12(N)CC3CC(C1)CC(C2)C3,NC12CC3CC(CC(C3)C1)C2,-1.372708
1,2,Amiodarone,C1=CC=CC2=C1C(=C(O2)CCCC)C(=O)C3=CC(=C(OCCN(CC...,CCCCc1oc2ccccc2c1C(=O)c1cc(I)c(OCCN(CC)CC)c(I)c1,-9.061896
2,3,Amodiaquine,C1=CC(=CC2=NC=CC(=C12)NC3=CC=C(C(=C3)CN(CC)CC)...,CCN(CC)Cc1cc(Nc2ccnc3cc(Cl)ccc23)ccc1O,-5.576931
3,4,Bisoprolol,CC(C)NCC(COc1ccc(cc1)COCCOC(C)C)O,CC(C)NCC(O)COc1ccc(COCCOC(C)C)cc1,-2.095795
4,5,Bromocriptine,[C@@]56(N(C([C@@](NC([C@@H]4C=C3C1=CC=CC2=C1C(...,CC(C)CC1C(=O)N2CCCC2C2(O)OC(NC(=O)C3C=C4c5cccc...,-6.067071
5,6,Buprenorphine,O[C@]([C@@H]1([C@]2([C@H]3([C@]45([C@@]([C@H](...,COC12CCC3(CC1C(C)(O)C(C)(C)C)C1Cc4ccc(O)c5c4C3...,-5.732224
6,7,Chlorprothixene,C1=C(Cl)C=CC3=C1\C(C2=C(C=CC=C2)S3)=C/CCN(C)C,CN(C)CCC=C1c2ccccc2Sc2ccc(Cl)cc21,-6.024933
7,8,Clofazimine,C1=CC=CC3=C1N(C2=CC(=NC(C)C)C(=CC2=N3)NC4=CC=C...,CC(C)N=c1cc2n(-c3ccc(Cl)cc3)c3ccccc3nc-2cc1Nc1...,-6.296282
8,9,Curcumin,Oc1ccc(cc1OC)/C=C/C(=O)CC(=O)/C=C/c2ccc(O)c(OC)c2,COc1cc(C=CC(=O)CC(=O)C=Cc2ccc(O)c(OC)c2)ccc1O,-4.697997
9,10,Danazol,[C@@H]23[C@H]([C@H]1[C@]([C@@](C#C)(O)CC1)(C)C...,C#CC1(O)CCC2C3CCC4=Cc5oncc5CC4(C)C3CCC21C,-5.541718


In [87]:
def calc_mol_weight(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        return Descriptors.MolWt(mol)
    return None

# Apply the function to the SMILES column
df_set1['MW'] = df_set1['SMILES'].apply(calc_mol_weight)

In [88]:
df_set2['MW'] = df_set2['SMILES'].apply(calc_mol_weight)

In [89]:
df_set1['Sol_g_l']=(10**df_set1.Prediction)*df_set1.MW
df_set2['Sol_g_l']=(10**df_set2.Prediction)*df_set2.MW

In [93]:
df_set1

,Sr No,COMPOUND_Name,SMILES,smiles_canon,Prediction,MW,Sol_g_l
0,1,Acetazolamide,CC(NC1=NN=C(S1)[S](N)(=O)=O)=O,CC(=O)Nc1nnc(S(N)(=O)=O)s1,-1.285938,222.251,11.505518
1,2,Acetylsalicylic_Acid,C(C)(=O)OC1=CC=CC=C1C(=O)O,CC(=O)Oc1ccccc1C(=O)O,-1.640043,180.159,4.126797
2,3,Alclofenac,C=CCOc1ccc(cc1Cl)CC(=O)O,C=CCOc1ccc(CC(=O)O)cc1Cl,-2.987961,226.659,0.233030
3,4,Ambroxol,O[C@@H]2CC[C@@H](NCc1cc(Br)cc(Br)c1N)CC2,Nc1c(Br)cc(Br)cc1CNC1CCC(O)CC1,-3.662070,378.108,0.082328
4,5,Aripiprazole,O=C1Nc2c(ccc(OCCCCN3CCN(c4c(Cl)c(Cl)ccc4)CC3)c...,O=C1CCc2ccc(OCCCCN3CCN(c4cccc(Cl)c4Cl)CC3)cc2N1,-6.012181,448.394,0.000436
...,...,...,...,...,...,...,...
95,96,Thiacetazone,C1=C(C=CC(=C1)NC(=O)C)/C=N/NC(=S)N,CC(=O)Nc1ccc(C=NNC(N)=S)cc1,-2.331076,236.300,1.102523
96,97,Triamcinolone,[C@]23([C@H]([C@H]1[C@]([C@](C(CO)=O)(O)[C@@H]...,CC12C=CC(=O)C=C1CCC1C3CC(O)C(O)(C(=O)CO)C3(C)C...,-3.430126,394.439,0.146506
97,98,Triamterene,C3=C(C1=C(N=C2C(=N1)C(=NC(=N2)N)N)N)C=CC=C3,Nc1nc(N)c2nc(-c3ccccc3)c(N)nc2n1,-3.587838,253.269,0.065425
98,99,Warfarin,C3=C(C(C1=C(OC2=C(C1=O)C=CC=C2)O)CC(=O)C)C=CC=C3,CC(=O)CC(c1ccccc1)c1c(O)oc2ccccc2c1=O,-4.412152,308.333,0.011936


In [91]:
### saving the predicted csv file to the disk 

df_set1.to_csv("JCIM_set1_prediction.csv", index=False)  ### 100 compounds 
df_set2.to_csv("JCIM_set2_prediction.csv", index=False)  ### 32 compounds 